# OptiVision measurement batch 11_text: the text models, on BEIR SciFact

Measures 14 models one after the other with the package's own loaders, then zips
the results. Each model is checked on 12 pages first (`check_retriever`); a model that fails
the check or a step is recorded and the batch goes on.

- `NeuML/colbert-bert-tiny`
- `mixedbread-ai/mxbai-edge-colbert-v0-17m`
- `mixedbread-ai/mxbai-edge-colbert-v0-32m`
- `lightonai/colbertv2.0`
- `colbert-ir/colbertv2.0`
- `lightonai/GTE-ModernColBERT-v1`
- `lightonai/Reason-ModernColBERT`
- `lightonai/ColBERT-Zero`
- `lightonai/LateOn`
- `lightonai/mLateOn`
- `mixedbread-ai/mxbai-colbert-large-v1`
- `LiquidAI/LFM2-ColBERT-350M`
- `LiquidAI/LFM2.5-ColBERT-350M`
- `jinaai/jina-colbert-v2`

**Run it:** *Settings -> Accelerator*: **GPU T4 x2**, *Internet*: **On**. Then
*Save Version -> Save & Run All (Commit)* and close the tab; it runs on its own for up to 12 h.
Download `results_11_text.zip` from the version's *Output* tab and give it to Claude.

**Remote code:** some models here run Python from their own repositories inside this session (`trust_remote_code`).

In [ ]:
BRANCH = "kaggle-campaign"
BATCH = [
 {
  "model": "NeuML/colbert-bert-tiny",
  "kind": "text"
 },
 {
  "model": "mixedbread-ai/mxbai-edge-colbert-v0-17m",
  "kind": "text"
 },
 {
  "model": "mixedbread-ai/mxbai-edge-colbert-v0-32m",
  "kind": "text"
 },
 {
  "model": "lightonai/colbertv2.0",
  "kind": "text"
 },
 {
  "model": "colbert-ir/colbertv2.0",
  "kind": "text"
 },
 {
  "model": "lightonai/GTE-ModernColBERT-v1",
  "kind": "text"
 },
 {
  "model": "lightonai/Reason-ModernColBERT",
  "kind": "text"
 },
 {
  "model": "lightonai/ColBERT-Zero",
  "kind": "text"
 },
 {
  "model": "lightonai/LateOn",
  "kind": "text"
 },
 {
  "model": "lightonai/mLateOn",
  "kind": "text"
 },
 {
  "model": "mixedbread-ai/mxbai-colbert-large-v1",
  "kind": "text"
 },
 {
  "model": "LiquidAI/LFM2-ColBERT-350M",
  "kind": "text"
 },
 {
  "model": "LiquidAI/LFM2.5-ColBERT-350M",
  "kind": "text",
  "trust_remote_code": true
 },
 {
  "model": "jinaai/jina-colbert-v2",
  "kind": "text",
  "trust_remote_code": true
 }
]
EXTRA_PIP = ["sentence-transformers[image]>=6.0.0", "einops"]
DEADLINE_HOURS = 11.0
SELECTION = True   # also run the selection study for a model when the session has time to spare

In [ ]:
import json, os, subprocess, sys, time
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
WORK = "/kaggle/working"
REPO = f"{WORK}/optivision-rag"
NAME = "11_text"
RESULTS = f"{WORK}/results_{NAME}"
os.environ["HF_HOME"] = "/tmp/hf"
!rm -rf {REPO} && git clone -q --depth 1 --branch {BRANCH} https://github.com/Daemon-VI/optivision-rag.git {REPO}
!cd {REPO} && git log --oneline -1
!pip install -q -e "{REPO}[bench,merge,vlm]" 2>&1 | tail -2
!pip install -q -U colpali-engine 2>&1 | tail -1
for package in EXTRA_PIP:
    !pip install -q "{package}" 2>&1 | tail -1
# Kaggle preinstalls torchao 0.10; peft refuses to work while an incompatible torchao is importable
!pip uninstall -y -q torchao
os.makedirs(RESULTS, exist_ok=True)
json.dump(BATCH, open(f"{WORK}/batch.json", "w"), indent=1)

In [ ]:
cmd = [sys.executable, f"{REPO}/scripts/measure_models.py", f"{WORK}/batch.json", "--out", RESULTS,
       "--data", "/tmp/optdata", "--deadline-hours", str(DEADLINE_HOURS)] + (["--selection"] if SELECTION else [])
subprocess.run(cmd, check=False)

In [ ]:
!pip freeze > {RESULTS}/pip_freeze.txt
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > {RESULTS}/gpu.txt
!cd {REPO} && git rev-parse HEAD > {RESULTS}/commit.txt
!cat {RESULTS}/batch_summary.json
!cd {WORK} && rm -f results_{NAME}.zip && zip -q -r results_{NAME}.zip results_{NAME} && ls -la results_{NAME}.zip